# Multi-Agent Research Report Generator (CrewAI + Gemini)

This notebook builds an automated "newsroom" of AI agents that work together to research, write, fact-check, humanize, combine, and publish a long-form report on a topic of your choice.

We use the **CrewAI** framework, which lets you define a team of `Agent`s (each with a role, a goal, and a backstory that shapes how it "thinks"), give them `Task`s to complete, and wire those tasks together into a `Crew` that runs them in order.

The pipeline we build has **ten agents**:

1. A **Research & Evidence Analyst** who gathers facts, statistics, recent examples, and case studies from the model's training knowledge.
2. A **Web Research & Fact Verification Specialist** who uses live Tavily web search to verify facts, surface recent (2023–2025) examples, and find attributed quotes for the introduction and conclusion.
3. An **Introduction Writer** who opens the report with the retrieved quote and frames the topic for the reader.
4. A **Core Argument Analyst** who states the report's central thesis.
5. A **Multi-Dimensional Analyst** who explores the issue from several angles (environmental, economic, social, political, etc.).
6. A **Counter-Argument & Critique Specialist** who raises and answers objections.
7. A **Synthesis & Conclusion Writer** who ties everything together, makes recommendations, and closes with the retrieved conclusion quote.
8. A **Fact-Checker & Humanizing Editor** who independently reviews *each* of the sections above and rewrites them using the style rules in `skills/SKILL.md`.
9. A **Managing Editor** who combines all the verified sections into one cohesive report with a `final_word_count`-limited executive summary.
10. A **Publishing Specialist** who saves the final report as a Word (`.docx`) document using a custom tool.

As a worked example, we'll ask the crew to write a report on **environmental security, climate-change-driven disasters, and their consequences, with a particular focus on India** — but you can change the topic in the "Report inputs" section below and rerun everything.

By default this notebook uses **Google Gemini** models as the underlying LLMs, assigning different model tiers to different agents to manage token costs. Groq models are used for the fastest, simplest tasks. All models are configured in Step 4.

## 1. Prerequisites

This notebook depends on a few Python packages on top of the ones already used in this project:

- `crewai` — the multi-agent orchestration framework (already a project dependency).
- `google-genai` — Google's Gen AI SDK, which CrewAI uses under the hood to talk to Gemini models.
- `python-docx` — used by our custom tool to write the final report out as a `.docx` Word document.
- `python-dotenv` — used to load API keys from the `.env` file (already a project dependency).
- `tavily-python` — the official Tavily client, used by the Web Research agent to make live web searches.

You need the following keys in your `.env` file:

| Key | Required for |
|-----|--------------|
| `GOOGLE_API_KEY` | All Gemini models (default LLMs) |
| `GROQ_API_KEY` | Groq models (used for the doc-writer agent) |
| `TAVILY_API_KEY` | Live web search in the Web Research agent |
| `OPENAI_API_KEY` | Only needed if you switch to an OpenAI model |

Run the cell below once to install any missing packages. If they are already installed, `pip` will confirm that the requirement is satisfied.

In [ ]:
%pip install -q google-genai python-docx tavily-python

## 2. Imports

Here is what each import is for:

- `os` and `pathlib.Path` — reading environment variables and working with file paths (for the `.env` file, the skill file, and the output `.docx` file).
- `dotenv.load_dotenv` — loads the key/value pairs from your `.env` file into the process's environment variables.
- `crewai.Agent`, `Task`, `Crew`, `Process`, `LLM` — the core building blocks: `Agent` defines a persona, `Task` defines a unit of work, `Crew` groups everything and runs the pipeline, `Process` controls the running order, and `LLM` wraps whichever language model we choose.
- `crewai.tools.tool` — a decorator that turns an ordinary Python function into a tool an agent can call.
- `docx.Document` — the class from `python-docx` that represents a Word document we can build up and save.
- `IPython.display.Markdown` and `display` — used at the end of the notebook to render the final report as formatted text.
- `tavily.TavilyClient` — the Tavily search client used by the Web Research agent to query the live web.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from docx import Document
from IPython.display import Markdown, display
from tavily import TavilyClient

## 3. Load environment variables and API keys

`load_dotenv(override=True)` reads the `.env` file in this project's root folder and copies its values into the environment variables that this Python process can see. Passing `override=True` means that if a variable is already set elsewhere (for example in your shell), the value from `.env` will take priority — this keeps the notebook's behaviour predictable.

We read four API keys:

- **`GOOGLE_API_KEY`** — required; powers all Gemini models used across the pipeline.
- **`GROQ_API_KEY`** — optional but recommended; used by `llm_groq_fast` for the Publishing Specialist.
- **`TAVILY_API_KEY`** — required for the Web Research agent's live web searches. Without it, that agent will skip searches and return a warning.
- **`OPENAI_API_KEY`** — loaded for convenience in case you want to swap in an OpenAI model.

In [2]:
load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key   = os.getenv("GROQ_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not google_api_key:
    raise ValueError(
        "GOOGLE_API_KEY is missing. Add it to your .env file before running this notebook."
    )

print(f"Google API key loaded  (starts with {google_api_key[:8]}...)")

if groq_api_key:
    print(f"Groq API key loaded    (starts with {groq_api_key[:8]}...)")
else:
    print("Warning: GROQ_API_KEY not found — Groq models will not be available.")

if tavily_api_key:
    print(f"Tavily API key loaded  (starts with {tavily_api_key[:8]}...)")
else:
    print("Warning: TAVILY_API_KEY not found — web research agent will not function.")

Google API key loaded  (starts with AQ.Ab8RN...)
Groq API key loaded    (starts with gsk_9sQd...)
Tavily API key loaded  (starts with tvly-dev...)


## 4. Choose the language models

Rather than giving every agent the same model, we define five named `LLM` objects and assign them to agents based on their workload:

| Variable | Model | Assigned to |
|---|---|---|
| `llm_gemini_3_5` | `gemini/gemini-3.5-flash` | Creative/complex writers (intro, multi-dim, synthesis, combiner) |
| `llm_gemini_1_5` | `gemini/gemini-1.5-flash` | Factual/repeated tasks (research, core arg, counter-arg, verifier ×6) |
| `llm_gemini_2_flash` | `gemini/gemini-2.0-flash` | Tool-heavy web-research agent (fast, low latency) |
| `llm_groq_3_3` | `groq/llama-3.3-70b-versatile` | General Groq fallback |
| `llm_groq_fast` | `groq/llama-3.1-8b-instant` | Publishing Specialist (trivial save-to-disk task) |

`temperature` controls how creative vs. predictable the output is. We use `0.3` for the Gemini 3.5 creative writers and `0.1–0.2` for more deterministic factual tasks.

To swap an agent to a different provider, change its `llm=` argument to any of the five objects above — or define a new `LLM(...)` for OpenAI, Groq, or another CrewAI-supported provider.

In [3]:
GEMINI_MODEL = "gemini/gemini-3.5-flash"

# --- Primary models (Gemini via Google) ---

# Highest-capability Gemini: used for complex creative and analytical writing tasks.
llm_gemini_3_5 = LLM(
    model=GEMINI_MODEL,
    api_key=google_api_key,
    temperature=0.3,
)

# Established mid-tier Gemini: used for factual retrieval and tasks that run many times
# (e.g. the verifier/humanizer which is called once per section).
llm_gemini_1_5 = LLM(
    model="gemini/gemini-1.5-flash",
    api_key=google_api_key,
    temperature=0.2,
)

# Fast, efficient Gemini 2.0: used for tool-heavy tasks (web research) where speed
# matters more than deep reasoning.
llm_gemini_2_flash = LLM(
    model="gemini/gemini-2.0-flash",
    api_key=google_api_key,
    temperature=0.2,
)

# --- Secondary models (Groq — fast inference, requires GROQ_API_KEY) ---

# Large Groq model: used for general-purpose tasks where low latency is preferred.
llm_groq_3_3 = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0.2,
)

# Lightweight Groq model: ultra-fast; used for simple, low-stakes tasks such as
# handing a finished document to the save tool.
llm_groq_fast = LLM(
    model="groq/llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0.1,
)

print("Models configured:")
print(f"  llm_gemini_3_5    → {GEMINI_MODEL}               (complex analysis, creative writing)")
print( "  llm_gemini_1_5    → gemini/gemini-1.5-flash       (factual tasks, repeated calls)")
print( "  llm_gemini_2_flash→ gemini/gemini-2.0-flash       (tool-use / web research)")
print( "  llm_groq_3_3      → groq/llama-3.3-70b-versatile  (fast general tasks)")
print( "  llm_groq_fast     → groq/llama-3.1-8b-instant     (ultra-fast simple tasks)")

Models configured:
  llm_gemini_3_5    → gemini/gemini-3.5-flash               (complex analysis, creative writing)
  llm_gemini_1_5    → gemini/gemini-1.5-flash       (factual tasks, repeated calls)
  llm_gemini_2_flash→ gemini/gemini-2.0-flash       (tool-use / web research)
  llm_groq_3_3      → groq/llama-3.3-70b-versatile  (fast general tasks)
  llm_groq_fast     → groq/llama-3.1-8b-instant     (ultra-fast simple tasks)


## 5. Define the report's topic, audience, and tone

CrewAI lets you write `Task` descriptions as templates containing placeholders like `{topic}`, and then fill them in later by passing an `inputs` dictionary to `crew.kickoff(...)`. This means we can write all of our task descriptions once, using `{topic}`, `{audience}`, and `{tone}`, and then generate a report on a *different* subject just by changing the dictionary below and re-running the crew - no need to edit every task.

For our worked example we'll focus on **environmental security and climate-change-driven disasters in India**, but feel free to change `topic` to anything you like.

In [4]:
report_inputs = {
    "topic": "Environmental Security in India",
    "audience": "policy analysts, journalists, and informed general readers",
    "tone": "clear, engaging, and natural, never robotic or repetitive",
    "context": "AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security",

    # ── Word-count limits substituted into task templates at crew.kickoff() time ──
    "research_task_word_limit":   300,   # evidence dossier
    "intro_word_limit":           300,   # introduction section
    "core_word_limit":            150,   # core argument section
    "multi_dimension_word_limit": 700,   # multi-dimensional discussion
    "counter_word_limit":         200,   # counter-arguments section
    "conclusion_word_limit":      200,   # synthesis & conclusion section
    "combined_word_limit":       1500,   # full assembled report
    "final_word_count":           200,   # executive summary in the published report
}

report_inputs

{'topic': 'Environmental Security in India',
 'audience': 'policy analysts, journalists, and informed general readers',
 'tone': 'clear, engaging, and natural, never robotic or repetitive',
 'context': 'AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security',
 'research_task_word_limit': 300,
 'intro_word_limit': 300,
 'core_word_limit': 150,
 'multi_dimension_word_limit': 700,
 'counter_word_limit': 200,
 'conclusion_word_limit': 200,
 'combined_word_limit': 1500,
 'final_word_count': 200}

## 6. Load the humanizer skill instructions

The "Fact-Checker & Humanizing Editor" agent we'll create later needs to know *how* to humanize text. Rather than hard-coding those rules into this notebook, we keep them in a separate Markdown file, `skills/humanizer/SKILL.md`, which describes both a fact-checking checklist and a list of writing-style rules for making AI-drafted text sound more natural.

Keeping these instructions in their own file means you can tune the editor's behaviour - for example, make it stricter about fact-checking, or change the target writing style - without touching this notebook at all. Here we simply read the file's contents into a string so we can embed it in that agent's `backstory` further down.

In [5]:
humanizer_skill_path = Path("skills/SKILL.md")
humanizer_instructions = humanizer_skill_path.read_text(encoding="utf-8")

print(f"Loaded {len(humanizer_instructions)} characters of humanizer instructions from {humanizer_skill_path}")
print("---")
print(humanizer_instructions[:400] + "...")

Loaded 27675 characters of humanizer instructions from skills\SKILL.md
---
---
name: humanizer
version: 2.5.1
description: |
  Remove signs of AI-generated writing from text. Use when editing or reviewing
  text to make it sound more natural and human-written. Based on Wikipedia's
  comprehensive "Signs of AI writing" guide. Detects and fixes patterns including:
  inflated symbolism, promotional language, superficial -ing analyses, vague
  attributions, em dash overuse, ...


## 7. Custom tools: Word document writer and web search

Agents in CrewAI can only directly produce *text*. We give two agents access to **tools** — ordinary Python functions decorated with `@tool` — so they can interact with the outside world.

### Tool 1 — `save_report_as_word_document`

Given to the **Publishing Specialist**. It walks through the final Markdown report line by line and converts it to a formatted `.docx` file using `python-docx`:

- Lines starting with `#`, `##`, or `###` become Word headings.
- Lines starting with `-` or `*` become bulleted list items.
- Lines starting with `>` get the "Intense Quote" paragraph style.
- Any other non-empty line becomes a normal paragraph.
- Text wrapped in `**double asterisks**` is rendered in bold.

### Tool 2 — `tavily_web_search`

Given to the **Web Research & Fact Verification Specialist**. It calls the Tavily API with `search_depth="basic"` and returns the top 3 results (title, URL, and a ≤ 500-character snippet) per query. The tool's docstring instructs the agent to make no more than 4 searches total, and `max_iter=5` on the agent enforces a hard cap on reasoning rounds to protect the free-tier quota.

In [6]:
@tool("save_report_as_word_document")
def save_report_as_word_document(report_markdown: str, file_name: str) -> str:
    """Save a research report written in simple Markdown as a formatted Microsoft Word (.docx) file.

    The report_markdown text may use '#', '##' and '###' at the start of a line for
    headings, '-' or '*' for bullet points, '>' for a short quoted note, and
    '**bold text**' for bold emphasis. Every other non-empty line is treated as a
    normal paragraph.

    Args:
        report_markdown: The full text of the report, in Markdown format.
        file_name: The file name to save the document as, e.g. 'report.docx'.

    Returns:
        A short confirmation message containing the absolute path of the saved file.
    """
    document = Document()

    def add_text_with_bold(paragraph, text):
        # Splitting on "**" means every second chunk (index 1, 3, 5, ...) was
        # originally wrapped in "**bold**" markers, so we render it as bold.
        for index, chunk in enumerate(text.split("**")):
            if chunk == "":
                continue
            run = paragraph.add_run(chunk)
            run.bold = index % 2 == 1

    for raw_line in report_markdown.splitlines():
        line = raw_line.strip()

        if not line:
            continue
        elif line.startswith("### "):
            document.add_heading(line[4:].strip(), level=3)
        elif line.startswith("## "):
            document.add_heading(line[3:].strip(), level=2)
        elif line.startswith("# "):
            document.add_heading(line[2:].strip(), level=1)
        elif line.startswith("- ") or line.startswith("* "):
            paragraph = document.add_paragraph(style="List Bullet")
            add_text_with_bold(paragraph, line[2:].strip())
        elif line.startswith(">"):
            paragraph = document.add_paragraph(style="Intense Quote")
            add_text_with_bold(paragraph, line.lstrip("> ").strip())
        else:
            paragraph = document.add_paragraph()
            add_text_with_bold(paragraph, line)

    output_path = Path(file_name).resolve()
    document.save(output_path)

    return f"Report saved to {output_path}"

In [7]:
@tool("tavily_web_search")
def tavily_web_search(query: str) -> str:
    """Search the web for recent news, facts, examples, and notable quotes on a topic.

    IMPORTANT – quota conservation rules (follow strictly):
    - Perform a MAXIMUM of 4 searches for your entire task. Plan all queries before
      making the first call, so no quota is wasted on redundant lookups.
    - Use broad, well-targeted queries that surface multiple useful results in one call.
    - Do NOT repeat semantically similar queries.

    Args:
        query: A specific, focused search query
               (e.g. 'India groundwater depletion 2024 latest statistics').

    Returns:
        Top 3 search results: title, source URL, and a content snippet (≤ 500 chars each).
    """
    if not tavily_api_key:
        return "Tavily API key not configured. Cannot perform web search."

    client = TavilyClient(api_key=tavily_api_key)
    response = client.search(query, max_results=3, search_depth="basic")

    results = response.get("results", [])
    if not results:
        return "No results found for this query."

    formatted = []
    for r in results:
        snippet = r.get("content", "")[:500]
        formatted.append(
            f"**{r.get('title', 'Untitled')}**\n"
            f"Source: {r.get('url', 'N/A')}\n"
            f"{snippet}"
        )
    return "\n\n---\n\n".join(formatted)

## 8. Defining the agents

Every `Agent` is configured through three fields that shape the prompt the model receives:

- **`role`** — a short job title (e.g. "Introduction Writer").
- **`goal`** — what the agent is trying to achieve, written as an instruction.
- **`backstory`** — context that shapes the agent's voice, priorities, and decision-making. This is also where we embed the humanizer skill instructions for the Fact-Checker agent.

Each agent is assigned a specific `llm` object (see Step 4) to balance output quality against token cost. Setting `verbose=True` on every agent means you can watch their reasoning in the notebook output as the crew runs.

### Agents 1–6: the section writers

The first six agents each own one section of the final report. Their `goal` and `backstory` have been updated to reflect the web-research integration: the Introduction Writer opens with the Intro Quote, and the Synthesis & Conclusion Writer closes with the Conclusion Quote.

### Agent 7: the Fact-Checker & Humanizing Editor

This agent has a dual job: it cross-checks the claims in each section against the evidence the research agent gathered, and it rewrites the section so it doesn't read like typical AI output.

Rather than repeating those rules in every task description, we embed the entire contents of `humanizer_instructions` (loaded in Step 6 from `skills/SKILL.md`) directly into this agent's `backstory`. Because this happens once at agent creation time, the same instructions are automatically available no matter which section this agent is asked to review. We reuse this single agent for **six** separate verification tasks — one per section — which is why it is assigned the mid-tier `llm_gemini_1_5` model to conserve token quota.

In [ ]:
fact_finder_agent = Agent(
    role="Research & Evidence Analyst",
    goal=(
        "Compile accurate, concrete facts, statistics, recent real-world examples, and "
        "case studies on the report's topic, clearly noting approximate dates, locations, "
        "and sources so that the other writers can rely on them."
    ),
    backstory=(
        "You have spent years gathering data on environmental and disaster-related issues "
        "for policy reports. You favour credible sources such as government and UN agency "
        "reports, peer-reviewed studies, and reputable news coverage, and you always note "
        "roughly when and where something happened so claims can be checked later."
    ),
    llm=llm_gemini_1_5,
    verbose=True,
)

intro_writer_agent = Agent(
    role="Introduction Writer",
    goal=(
        "Write an engaging introduction that frames the report's topic, explains why it "
        "matters right now, previews what the rest of the report will cover, and "
        "opens with the compelling quote provided by the web research agent."
    ),
    backstory=(
        "You open long-form reports for a living. Readers decide in the first paragraph "
        "whether to keep reading, so you hook them with a concrete detail or a striking "
        "quote before zooming out to the bigger picture."
    ),
    llm=llm_gemini_3_5,
    verbose=True,
)

core_argument_agent = Agent(
    role="Core Argument Analyst",
    goal=(
        "State the report's central thesis clearly and lay out the main line of reasoning "
        "and evidence that supports it."
    ),
    backstory=(
        "You think in terms of claim, reasons, and evidence. You are good at distilling a "
        "complex issue down to one clear, defensible thesis and explaining why it holds up."
    ),
    llm=llm_gemini_1_5,
    verbose=True,
)

multi_dimension_agent = Agent(
    role="Multi-Dimensional Analyst",
    goal=(
        "Show how the report's core argument plays out across several different "
        "dimensions - for example environmental, economic, social or public-health, and "
        "political or governance dimensions - so readers see the full picture."
    ),
    backstory=(
        "You specialise in systems thinking: you map how one issue ripples outward into "
        "other areas of life, and you organise that complexity into clearly labelled "
        "sections so it stays easy to follow."
    ),
    llm=llm_gemini_3_5,
    verbose=True,
)

counter_argument_agent = Agent(
    role="Counter-Argument & Critique Specialist",
    goal=(
        "Present the strongest good-faith objections or alternative viewpoints to the "
        "report's core argument, and respond to each of them fairly."
    ),
    backstory=(
        "You play devil's advocate professionally. You state the opposing view as "
        "persuasively as its proponents would, and then give a measured, evidence-based "
        "response rather than a dismissive one."
    ),
    llm=llm_gemini_1_5,
    verbose=True,
)

synthesis_conclusion_agent = Agent(
    role="Synthesis & Conclusion Writer",
    goal=(
        "Pull together the introduction, core argument, multi-dimensional analysis, "
        "evidence, and counter-arguments into one coherent synthesis, restate the overall "
        "takeaway in your own words, end with concrete recommendations, and close with "
        "the forward-looking quote provided by the web research agent."
    ),
    backstory=(
        "You write the closing section of reports. You are good at noticing the thread "
        "that runs through everything that came before and turning it into a memorable, "
        "actionable ending that leaves the reader with a clear call to action."
    ),
    llm=llm_gemini_3_5,
    verbose=True,
)

### Agent 8: the Managing Editor

Once every section has been individually verified and humanized, something still needs to stitch them into one document with a consistent voice, smooth transitions, and a title and executive summary. That is this agent's job. It is assigned `llm_gemini_3_5` because the combine step demands the highest editorial quality in the pipeline.

In [ ]:
verifier_humanizer_agent = Agent(
    role="Fact-Checker & Humanizing Editor",
    goal=(
        "Independently check each section of the report for factual accuracy against the "
        "evidence gathered, then rewrite it so it reads naturally and avoids robotic, "
        "repetitive AI-style phrasing."
    ),
    backstory=(
        "You are a meticulous copy-editor and fact-checker. You always cross-check claims "
        "against the evidence you are given before changing anything else, and you follow "
        "the house style guide below very closely whenever you rewrite a section.\n\n"
        f"{humanizer_instructions}"
    ),
    llm=llm_gemini_1_5,
    verbose=True,
)

### Agent 9: the Publishing Specialist

The final agent before the Web Research agent's step (which was defined earlier) has one job: hand the finished report to the `save_report_as_word_document` tool and return the confirmation message. It is assigned `llm_groq_fast` — the lightest model available — because the task requires no reasoning, only a single tool call.

In [ ]:
verifier_humanizer_agent = Agent(
    role="Fact-Checker & Humanizing Editor",
    goal=(
        "Independently check each section of the report for factual accuracy against the "
        "evidence gathered, then rewrite it so it reads naturally and avoids robotic, "
        "repetitive AI-style phrasing."
    ),
    backstory=(
        "You are a meticulous copy-editor and fact-checker. You always cross-check claims "
        "against the evidence you are given before changing anything else, and you follow "
        "the house style guide below very closely whenever you rewrite a section.\n\n"
        f"{humanizer_instructions}"
    ),
    llm=llm,
    verbose=True,
)

In [ ]:
combiner_agent = Agent(
    role="Managing Editor",
    goal=(
        "Combine the verified, humanized sections into one cohesive, well-structured "
        "Markdown report with smooth transitions, consistent terminology, and a single "
        "unified voice."
    ),
    backstory=(
        "You assemble individually-written sections into a finished report. You smooth "
        "over abrupt transitions, remove duplicated points, make sure headings and "
        "terminology are consistent throughout, and add a short title and a one-paragraph "
        "executive summary at the very top."
    ),
    llm=llm,
    verbose=True,
)

In [ ]:
doc_writer_agent = Agent(
    role="Publishing Specialist",
    goal=(
        "Take the final, combined report and use the available tool to save it as a "
        "polished Word document, then return the tool's confirmation message."
    ),
    backstory=(
        "You are the final step before a report goes out the door. You do not rewrite "
        "content - your job is to faithfully hand the finished Markdown report to the "
        "document-generation tool and confirm that the file was saved."
    ),
    tools=[save_report_as_word_document],
    llm=llm_groq_fast,
    verbose=True,
)

In [ ]:
web_researcher_agent = Agent(
    role="Web Research & Fact Verification Specialist",
    goal=(
        "Using targeted web searches, verify the key facts from the evidence dossier "
        "against live sources, surface the most recent (2023-2025) examples and "
        "statistics, and find one compelling attributed quote for the introduction "
        "and one forward-looking attributed quote for the conclusion. "
        "QUOTA LIMIT: make no more than 4 tavily_web_search calls in total. "
        "Plan all queries mentally before executing any of them."
    ),
    backstory=(
        "You are a seasoned investigative researcher with a nose for primary sources. "
        "You know how to write search queries that return maximum signal in a single call, "
        "so you never waste a lookup on something the first result already answered. "
        "You always record the source URL and approximate publication date of everything "
        "you retrieve, because your colleagues will cite it in a published report."
    ),
    tools=[tavily_web_search],
    llm=llm_gemini_2_flash,   # tool-heavy task — fast model keeps latency and cost low
    max_iter=5,                # hard cap on reasoning rounds to protect quota
    verbose=True,
)

## 9. Defining the tasks

A `Task` describes one unit of work: what to do (`description`), what a good result looks like (`expected_output`), which `agent` should do it, and which earlier tasks' outputs it needs as `context`. When a task lists other tasks in `context`, CrewAI automatically passes those outputs to the agent as additional information.

Every `description` and `expected_output` string uses `{placeholder}` syntax. CrewAI substitutes these from the `report_inputs` dictionary at `crew.kickoff()` time — this includes the topic, audience, tone, context, and all eight word-count limits defined in Step 5.

The task execution order is:

1. **`research_task`** — runs first, with no dependencies, and produces the evidence dossier.
2. **`web_research_task`** — runs second; takes the evidence dossier as context and uses Tavily to verify facts and retrieve the two attributed quotes.
3. **`intro_task`** — receives both the evidence dossier and web research output; opens with the Intro Quote.
4. **`core_argument_task`** — depends on the evidence dossier and introduction.
5. **`multi_dimension_task`** and **`counter_argument_task`** — each build on all prior section tasks.
6. **`synthesis_task`** — depends on all previous sections and the web research output; closes with the Conclusion Quote.

In [ ]:
research_task = Task(
    description=(
        "Research the topic: '{topic} in the context of {context}'. Compile a structured evidence dossier containing:\n"
        "1. Key facts and statistics from credible reports (for example IPCC, government "
        "ministries, UN agencies, or peer-reviewed studies).\n"
        "2. At least three recent real-world examples or disasters that illustrate the "
        "issue, each with an approximate date and location.\n"
        "3. One or two short case studies describing impacts and/or responses in more "
        "depth.\n"
        "Write with {audience} in mind, and for each item briefly note what it illustrates."
    ),
    expected_output=(
        "A Markdown document with three sub-sections titled 'Key Facts & Statistics', "
        "'Recent Examples', and 'Case Studies', each containing several concrete, "
        "specific bullet points. Strict limitation: Do not exceed {research_task_word_limit} words."
    ),
    agent=fact_finder_agent,
)

# ── NEW: Web Research & Fact Verification task ─────────────────────────────────
web_research_task = Task(
    description=(
        "Using targeted web searches, do the following for the report on '{topic}' "
        "in the context of {context}:\n\n"
        "1. **Fact Verification**: Cross-check 3-4 key claims from the evidence dossier "
        "against live web sources. Note any figures that need updating.\n"
        "2. **Recent Examples (2023-2025)**: Find 2-3 real-world events, statistics, "
        "or case studies published in 2023-2025 that directly illustrate the topic.\n"
        "3. **Intro Quote**: Find one vivid, authoritative quote (attributed to a named "
        "person, official body, or credible publication) that can open the Introduction "
        "with immediate impact.\n"
        "4. **Conclusion Quote**: Find one forward-looking or sobering quote suitable "
        "for closing the Conclusion on a strong note.\n\n"
        "QUOTA LIMIT: Use the tavily_web_search tool no more than 4 times total. "
        "Plan all queries before making the first call."
    ),
    expected_output=(
        "A structured Markdown document with four clearly labelled sections:\n"
        "- '### Verified Facts' — confirmed or corrected key statistics with source URLs.\n"
        "- '### Recent Examples (2023-2025)' — 2-3 concrete recent examples with dates "
        "and source URLs.\n"
        "- '### Intro Quote' — the chosen quote with author/source attribution and a "
        "one-line note on why it works as an opener.\n"
        "- '### Conclusion Quote' — the chosen quote with author/source attribution and "
        "a one-line note on why it works as a closer.\n"
        "Strict limitation: Do not exceed 300 words."
    ),
    agent=web_researcher_agent,
    context=[research_task],
)
# ───────────────────────────────────────────────────────────────────────────────

intro_task = Task(
    description=(
        "Using the evidence dossier and the web research output as background in the "
        "context of {context}, write the Introduction section of a report on '{topic}'. "
        "Open with the 'Intro Quote' provided by the web research agent — attribute it "
        "correctly — then explain what the issue is, why it matters right now, and "
        "briefly preview what the rest of the report will cover. "
        "Write in a {tone} tone for {audience}."
    ),
    expected_output=(
        "An Introduction section in Markdown (roughly 3-5 paragraphs), starting with the "
        "heading '## Introduction'. The first paragraph must open with the attributed "
        "quote from the web research output. "
        "Strict limitation: Do not exceed {intro_word_limit} words."
    ),
    agent=intro_writer_agent,
    context=[research_task, web_research_task],
)

core_argument_task = Task(
    description=(
        "Drawing on the evidence dossier, write the Core Argument section of the report "
        "on '{topic}' in the context of {context}. State the central thesis in one or two clear sentences near the "
        "start of the section, then explain the main reasoning and evidence that "
        "supports it. Write in a {tone} tone for {audience}."
    ),
    expected_output=(
        "A Core Argument section in Markdown starting with the heading '## Core "
        "Argument', containing a clearly stated thesis and supporting reasoning. "
        "Strict limitation: Do not exceed {core_word_limit} words."
    ),
    agent=core_argument_agent,
    context=[research_task, intro_task],
)

multi_dimension_task = Task(
    description=(
        "Building on the core argument and the evidence dossier, write a "
        "Multi-Dimensional Discussion section for the report on '{topic}' in the context of {context}. Discuss how "
        "the core argument plays out across several relevant dimensions - for example "
        "environmental, economic, social or public-health, agriculture, security and political or governance "
        "dimensions - using a clearly labelled sub-heading for each dimension. Write in "
        "a {tone} tone for {audience}."
    ),
    expected_output=(
        "A 'Multi-Dimensional Discussion' section in Markdown starting with '## "
        "Multi-Dimensional Discussion', containing at least three labelled sub-sections "
        "(use '### ' headings). "
        "Strict limitation: Do not exceed {multi_dimension_word_limit} words."
    ),
    agent=multi_dimension_agent,
    context=[research_task, intro_task, core_argument_task],
)

counter_argument_task = Task(
    description=(
        "Considering the core argument and the evidence dossier, write a "
        "Counter-Arguments section for the report on '{topic}' in the context of {context}. Present at least two of "
        "the strongest good-faith objections or alternative viewpoints, and follow each "
        "one with a brief, fair, evidence-based response. Write in a {tone} tone for "
        "{audience}."
    ),
    expected_output=(
        "A 'Counter-Arguments' section in Markdown starting with '## Counter-Arguments', "
        "containing at least two objection/response pairs. "
        "Strict limitation: Do not exceed {counter_word_limit} words."
    ),
    agent=counter_argument_agent,
    context=[research_task, core_argument_task, multi_dimension_task],
)

synthesis_task = Task(
    description=(
        "Read the introduction, core argument, multi-dimensional discussion, evidence "
        "dossier, counter-arguments, and web research output for the report on '{topic}' "
        "in the context of {context}. Write a Synthesis & Conclusion section that ties "
        "these threads together, restates the overall takeaway in your own words, ends "
        "with a short numbered list of two to four concrete recommendations, and closes "
        "with the 'Conclusion Quote' from the web research output — attributed correctly. "
        "Write in a {tone} tone for {audience}."
    ),
    expected_output=(
        "Generate a Markdown section titled exactly '## Conclusion'. "
        "The section must provide a concise synthesis of the key findings, arguments, "
        "and insights presented throughout the document. Avoid introducing new facts, "
        "evidence, examples, or arguments. Focus on integration, evaluation, and overall significance. "
        "Conclude with a clearly identifiable recommendations subsection. "
        "Present recommendations as a numbered sequence using the format "
        "'First, ...', 'Second, ...', 'Third, ...', and so on. "
        "After the recommendations, close with the attributed Conclusion Quote from the "
        "web research output. "
        "The entire Conclusion section, including recommendations and closing quote, "
        "must not exceed {conclusion_word_limit} words under any circumstances."
    ),
    agent=synthesis_conclusion_agent,
    context=[
        research_task,
        web_research_task,
        intro_task,
        core_argument_task,
        multi_dimension_task,
        counter_argument_task,
    ],
)

## 10. Defining the verification tasks

Next we define six review tasks — one for each section of the report, including the evidence dossier itself. Every task is assigned to the *same* `verifier_humanizer_agent`, but each gives it a different pair of inputs in its `context`: the evidence dossier (so it has something to fact-check against) and the specific section it should review.

Two of these tasks also receive the `web_research_task` output in their context:

- **`verify_intro_task`** — checks that the opening quote is correctly attributed and integrated.
- **`verify_synthesis_task`** — checks that the closing Conclusion Quote is correctly attributed and that the numbered recommendations are intact.

Because the full fact-checking checklist and humanizing style rules already live in the `verifier_humanizer_agent`'s `backstory`, the task descriptions here can stay short — they just tell the agent which section to work on and what heading to preserve.

In [ ]:
verify_evidence_task = Task(
    description=(
        "Review the evidence dossier you were given for the report on '{topic}' in the context of {context}. Check "
        "that the figures, dates, and named events are plausible and internally "
        "consistent. Then rewrite it following the fact-checking and humanizing rules in "
        "your backstory. Keep the same three sub-headings ('Key Facts & Statistics', "
        "'Recent Examples', 'Case Studies')."
    ),
    expected_output=(
        "A revised version of the evidence dossier in Markdown, with the same three "
        "sub-headings, written in natural prose and bullet points and free of robotic "
        "phrasing. "
        "Strict limitation: Do not exceed {research_task_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task],
)

verify_intro_task = Task(
    description=(
        "Review the Introduction section you were given for the report on '{topic}' in the context of {context}, "
        "cross-checking any factual claims against the evidence dossier and the verified "
        "web research output. Confirm that the opening quote is correctly attributed and "
        "naturally integrated. Then rewrite it following the fact-checking and humanizing "
        "rules in your backstory."
    ),
    expected_output=(
        "A revised Introduction section in Markdown, starting with '## Introduction', "
        "that opens with the correctly attributed quote from the web research output. "
        "Strict limitation: Do not exceed {intro_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, web_research_task, intro_task],
)

verify_core_argument_task = Task(
    description=(
        "Review the Core Argument section you were given for the report on '{topic}' in the context of {context}, "
        "cross-checking any factual claims against the evidence dossier. Then rewrite it "
        "following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output=(
        "A revised Core Argument section in Markdown, starting with '## Core Argument'. "
        "Strict limitation: Do not exceed {core_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, intro_task, core_argument_task],
)

verify_multi_dimension_task = Task(
    description=(
        "Review the Multi-Dimensional Discussion section you were given for the report "
        "on '{topic}' in the context of {context}, cross-checking any factual claims against the evidence dossier. "
        "Then rewrite it following the fact-checking and humanizing rules in your "
        "backstory. Keep its sub-headings."
    ),
    expected_output=(
        "A revised Multi-Dimensional Discussion section in Markdown, starting with '## "
        "Multi-Dimensional Discussion' and keeping its '### ' sub-headings. "
        "Strict limitation: Do not exceed {multi_dimension_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, multi_dimension_task],
)

verify_counter_argument_task = Task(
    description=(
        "Review the Counter-Arguments section you were given for the report on "
        "'{topic}' in the context of {context}, cross-checking any factual claims against the evidence dossier. Then "
        "rewrite it following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output=(
        "A revised Counter-Arguments section in Markdown, starting with '## Counter-Arguments'. "
        "Strict limitation: Do not exceed {counter_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, counter_argument_task],
)

verify_synthesis_task = Task(
    description=(
        "Review the Synthesis & Conclusion section you were given for the report on "
        "'{topic}' in the context of {context}, cross-checking any factual claims against the evidence dossier and "
        "the web research output. Confirm that the closing Conclusion Quote is correctly "
        "attributed and naturally integrated. Then rewrite it following the fact-checking "
        "and humanizing rules in your backstory. Keep its numbered recommendations and "
        "closing quote."
    ),
    expected_output=(
        "A revised Synthesis & Conclusion section in Markdown, starting with '## "
        "Conclusion', ending with numbered recommendations followed by the correctly "
        "attributed Conclusion Quote. "
        "Strict limitation: Do not exceed {conclusion_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, web_research_task, synthesis_task],
)

## 11. Defining the combine and publish tasks

The last two tasks finish the pipeline:

- The **combine task** takes all six verified, humanized sections as `context`, performs a final editorial pass (removing redundancy, smoothing transitions, enforcing a consistent voice), and assembles them into one publication-ready Markdown report. The Executive Summary it generates must not exceed `final_word_count` words. It also receives `web_research_task` directly so the Managing Editor can confirm that both attributed quotes are correctly placed — the Intro Quote opening the Introduction and the Conclusion Quote closing the Synthesis & Conclusion.

- The **document task** takes the finished Markdown report and passes it unchanged to the `save_report_as_word_document` tool, then returns the tool's confirmation message (including the absolute path of the saved `.docx` file).

In [ ]:
combine_task = Task(
    description=(
        "Combine all verified, fact-checked, and humanized sections into a single "
        "publication-ready Markdown report on '{topic}' within the context of "
        "{context}. The report should be written in a consistent {tone} tone and "
        "tailored for an audience including {audience}. \n\n"

        "Before combining, perform a final editorial review across all sections.\n"

        "Tasks to perform:\n"
        "- Eliminate repetition, overlap, and contradictory statements.\n"
        "- Ensure factual consistency across sections.\n"
        "- Ensure terminology, definitions, and concepts are used consistently.\n"
        "- Verify that evidence cited in later sections aligns with arguments made earlier.\n"
        "- Remove unsupported claims, unverifiable statements, and orphaned references.\n"
        "- Ensure all cited studies, reports, statistics, and case studies remain relevant "
        "to the final narrative.\n"
        "- Improve transitions between sections to create a seamless reading experience.\n"
        "- Maintain logical progression from introduction to conclusion.\n"
        "- Preserve analytical depth while improving readability.\n"
        "- Remove generic AI-style transitions and repetitive phrasing.\n"
        "- Harmonize writing style across sections so the report reads as if written by a "
        "single author.\n"
        "- Maintain high sentence-length variation and natural human writing patterns.\n"
        "- Ensure the Introduction opens with the attributed quote from the web research "
        "output, and the Conclusion closes with the attributed Conclusion Quote.\n\n"

        "Structure the report as follows:\n\n"

        "# [Compelling Report Title]\n\n"

        "## Executive Summary\n"
        "- A concise overview of the report's key arguments, findings, and implications. "
        "Must not exceed {final_word_count} words.\n\n"

        "## Introduction\n"
        "## Core Argument\n"
        "## Multi-Dimensional Discussion\n"
        "## Evidence & Case Studies\n"
        "## Counter-Arguments\n"
        "## Synthesis & Conclusion\n\n"

        "The title should be specific, informative, and reflective of the report's central "
        "theme. The conclusion should integrate the report's findings rather than merely "
        "summarising them. Recommendations must appear at the end in sequential format "
        "such as 'First, ...', 'Second, ...', 'Third, ...', and so on, followed by the "
        "closing Conclusion Quote."
    ),

    expected_output=(
        "A single, publication-ready Markdown report that begins with a '# ' title, "
        "followed by an Executive Summary of no more than {final_word_count} words, "
        "then the sections Introduction, Core Argument, Multi-Dimensional Discussion, "
        "Evidence & Case Studies, Counter-Arguments, and Synthesis & Conclusion in that "
        "exact order. The Introduction must open with the attributed Intro Quote; the "
        "Conclusion must close with the attributed Conclusion Quote after the numbered "
        "recommendations. The report must exhibit consistent terminology, coherent "
        "transitions, verified evidence, factual accuracy, strong analytical rigor, "
        "human-like writing quality, and a unified narrative voice. "
        "Strict limitation: Do not exceed {combined_word_limit} words in total."
    ),

    agent=combiner_agent,

    context=[
        verify_evidence_task,
        verify_intro_task,
        verify_core_argument_task,
        verify_multi_dimension_task,
        verify_counter_argument_task,
        verify_synthesis_task,
        web_research_task,
    ],
)

doc_task = Task(
    description=(
        "Take the final combined report exactly as written and use the "
        "save_report_as_word_document tool to save it as a Word document named "
        "'environmental_security_india_report.docx'. Pass the full report text to the "
        "tool without summarising, shortening, or rewriting it, and return the tool's "
        "confirmation message as your final answer."
    ),
    expected_output=(
        "The confirmation message returned by the save_report_as_word_document tool, "
        "including the absolute path of the saved .docx file."
    ),
    agent=doc_writer_agent,
    context=[combine_task],
)

In [ ]:
verify_evidence_task = Task(
    description=(
        "Review the evidence dossier you were given for the report on '{topic}' in the context of {context}. Check "
        "that the figures, dates, and named events are plausible and internally "
        "consistent. Then rewrite it following the fact-checking and humanizing rules in "
        "your backstory. Keep the same three sub-headings ('Key Facts & Statistics', "
        "'Recent Examples', 'Case Studies')."
    ),
    expected_output=(
        "A revised version of the evidence dossier in Markdown, with the same three "
        "sub-headings, written in natural prose and bullet points and free of robotic "
        "phrasing."
        "Strict limitation: Do not exceed {research_task_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task],
)

verify_intro_task = Task(
    description=(
        "Review the Introduction section you were given for the report on '{topic}' in the context of {context}, "
        "cross-checking any factual claims against the evidence dossier. Then rewrite it "
        "following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output=(
        "A revised Introduction section in Markdown, starting with '## Introduction'."
        "Strict limitation: Do not exceed {intro_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, intro_task],
)

verify_core_argument_task = Task(
    description=(
        "Review the Core Argument section you were given for the report on '{topic}' in the context of {context}, "
        "cross-checking any factual claims against the evidence dossier. Then rewrite it "
        "following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output=(
        "A revised Core Argument section in Markdown, starting with '## Core Argument'."
        "Strict limitation: Do not exceed {core_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, intro_task, core_argument_task],
)

verify_multi_dimension_task = Task(
    description=(
        "Review the Multi-Dimensional Discussion section you were given for the report "
        "on '{topic}' in the context of {context}, cross-checking any factual claims against the evidence dossier. "
        "Then rewrite it following the fact-checking and humanizing rules in your "
        "backstory. Keep its sub-headings."
    ),
    expected_output=(
        "A revised Multi-Dimensional Discussion section in Markdown, starting with '## "
        "Multi-Dimensional Discussion' and keeping its '### ' sub-headings."
        "Strict limitation: Do not exceed {multi_dimension_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, multi_dimension_task],
)

verify_counter_argument_task = Task(
    description=(
        "Review the Counter-Arguments section you were given for the report on "
        "'{topic}' in the context of {context}, cross-checking any factual claims against the evidence dossier. Then "
        "rewrite it following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output=(
        "A revised Counter-Arguments section in Markdown, starting with '## Counter-Arguments'."
        "Strict limitation: Do not exceed {counter_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, counter_argument_task],
)

verify_synthesis_task = Task(
    description=(
        "Review the Synthesis & Conclusion section you were given for the report on "
        "'{topic}' in the context of {context}, cross-checking any factual claims against the evidence dossier. Then "
        "rewrite it following the fact-checking and humanizing rules in your backstory. "
        "Keep its bulleted list of recommendations."
    ),
    expected_output=(
        "A revised Synthesis & Conclusion section in Markdown, starting with '## "
        "Synthesis & Conclusion' and ending with a bulleted list of recommendations."
        "Strict limitation: Do not exceed {conclusion_word_limit} words."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, synthesis_task],
)

## 12. Assembling the crew

A `Crew` brings together the agents and tasks we've defined and decides how to run them. We use `Process.sequential`, which runs the fifteen tasks one after another in the order given, passing each task's `context` outputs along as it goes.

The ten agents are listed in the order they first appear in the pipeline. Note that `verifier_humanizer_agent` handles six separate verification tasks (one per section) — CrewAI reuses the same agent object for each, keeping its full backstory (including the humanizer instructions) in context throughout.

In [ ]:
crew = Crew(
    agents=[
        fact_finder_agent,
        web_researcher_agent,
        intro_writer_agent,
        core_argument_agent,
        multi_dimension_agent,
        counter_argument_agent,
        synthesis_conclusion_agent,
        verifier_humanizer_agent,
        combiner_agent,
        doc_writer_agent,
    ],
    tasks=[
        research_task,
        web_research_task,
        intro_task,
        core_argument_task,
        multi_dimension_task,
        counter_argument_task,
        synthesis_task,
        verify_evidence_task,
        verify_intro_task,
        verify_core_argument_task,
        verify_multi_dimension_task,
        verify_counter_argument_task,
        verify_synthesis_task,
        combine_task,
        doc_task,
    ],
    process=Process.sequential,
    verbose=True,
)

In [ ]:
combine_task = Task(
    description=(
        "Combine all verified, fact-checked, and humanized sections into a single "
        "publication-ready Markdown report on '{topic}' within the context of "
        "{context}. The report should be written in a consistent {tone} tone and "
        "tailored for an audience including {audience}. \n\n"

        "Before combining, perform a final editorial review across all sections.\n"

        "Tasks to perform:\n"
        "- Eliminate repetition, overlap, and contradictory statements.\n"
        "- Ensure factual consistency across sections.\n"
        "- Ensure terminology, definitions, and concepts are used consistently.\n"
        "- Verify that evidence cited in later sections aligns with arguments made earlier.\n"
        "- Remove unsupported claims, unverifiable statements, and orphaned references.\n"
        "- Ensure all cited studies, reports, statistics, and case studies remain relevant "
        "to the final narrative.\n"
        "- Improve transitions between sections to create a seamless reading experience.\n"
        "- Maintain logical progression from introduction to conclusion.\n"
        "- Preserve analytical depth while improving readability.\n"
        "- Remove generic AI-style transitions and repetitive phrasing.\n"
        "- Harmonize writing style across sections so the report reads as if written by a "
        "single author.\n"
        "- Maintain high sentence-length variation and natural human writing patterns.\n\n"

        "Structure the report as follows:\n\n"

        "# [Compelling Report Title]\n\n"

        "Executive Summary\n"
        "- A concise overview of the report's key arguments, findings, and implications.\n\n"

        "## Introduction\n"
        "## Core Argument\n"
        "## Multi-Dimensional Discussion\n"
        "## Evidence & Case Studies\n"
        "## Counter-Arguments\n"
        "## Synthesis & Conclusion\n\n"

        "The title should be specific, informative, and reflective of the report's central "
        "theme. The executive summary should be concise yet substantive.\n\n"

        "The conclusion should integrate the report's findings rather than merely summarizing "
        "them. Recommendations must appear at the end in sequential format such as "
        "'First, ...', 'Second, ...', 'Third, ...', and so on."
    ),

    expected_output=(
        "A single, publication-ready Markdown report that begins with a '# ' title, "
        "followed by an executive summary and the sections Introduction, Core Argument, "
        "Multi-Dimensional Discussion, Evidence & Case Studies, Counter-Arguments, and "
        "Synthesis & Conclusion in that exact order. The report must exhibit consistent "
        "terminology, coherent transitions, verified evidence, factual accuracy, strong "
        "analytical rigor, human-like writing quality, and a unified narrative voice. "
        "The final Conclusion section must end with actionable recommendations formatted "
        "as 'First, ...', 'Second, ...', 'Third, ...', etc."
        "Strict limitation: Do not exceed {combined_word_limit} words."
    ),

    agent=combiner_agent,

    context=[
        verify_evidence_task,
        verify_intro_task,
        verify_core_argument_task,
        verify_multi_dimension_task,
        verify_counter_argument_task,
        verify_synthesis_task,
    ],
)

doc_task = Task(
    description=(
        "Take the final combined report exactly as written and use the "
        "save_report_as_word_document tool to save it as a Word document named "
        "'environmental_security_india_report.docx'. Pass the full report text to the "
        "tool without summarising, shortening, or rewriting it, and return the tool's "
        "confirmation message as your final answer."
    ),
    expected_output=(
        "The confirmation message returned by the save_report_as_word_document tool, "
        "including the absolute path of the saved .docx file."
    ),
    agent=doc_writer_agent,
    context=[combine_task],
)

In [ ]:
crew = Crew(
    agents=[
        fact_finder_agent,
        intro_writer_agent,
        core_argument_agent,
        multi_dimension_agent,
        counter_argument_agent,
        synthesis_conclusion_agent,
        verifier_humanizer_agent,
        combiner_agent,
        doc_writer_agent,
    ],
    tasks=[
        research_task,
        intro_task,
        core_argument_task,
        multi_dimension_task,
        counter_argument_task,
        synthesis_task,
        verify_evidence_task,
        verify_intro_task,
        verify_core_argument_task,
        verify_multi_dimension_task,
        verify_counter_argument_task,
        verify_synthesis_task,
        combine_task,
        doc_task,
    ],
    process=Process.sequential,
    verbose=True,
)

## 13. Running the crew

`crew.kickoff(inputs=report_inputs)` starts the pipeline. CrewAI substitutes every `{placeholder}` in task descriptions and expected outputs using the `report_inputs` dictionary we defined in Step 5 — this covers the topic, audience, tone, context, and all eight word-count limits.

The crew runs **fifteen tasks** in sequence:

1. `research_task` — evidence dossier
2. `web_research_task` — live fact-check + quotes (≤ 4 Tavily searches)
3. `intro_task` — introduction (opens with Intro Quote)
4. `core_argument_task` — central thesis
5. `multi_dimension_task` — multi-dimensional analysis
6. `counter_argument_task` — counter-arguments
7. `synthesis_task` — conclusion (closes with Conclusion Quote)
8–13. Six `verify_*` tasks — fact-check and humanize each section
14. `combine_task` — assemble final report with ≤ `final_word_count`-word exec summary
15. `doc_task` — save as `.docx`

Expect this to take several minutes. With `verbose=True` you'll see a running log of each agent's output as the crew works through the pipeline.

## 14. Viewing the results

`crew.kickoff(...)` returns a `CrewOutput` object. Its `.raw` attribute holds the text produced by the *last* task in the pipeline - in our case, that's the confirmation message from the Publishing Specialist's tool call, telling us where the `.docx` file was saved.

The full combined report (before it was handed to the document-writing tool) is the output of `combine_task`, which we can reach via `combine_task.output.raw`. We display both below: the tool's confirmation message, and the final report rendered as Markdown so you can read it directly in the notebook.

## 15. Ideas for extending this notebook

- **Change the topic**: edit `report_inputs` in Step 5 and re-run the notebook from that cell onwards — no agent or task definitions need to change.
- **Swap a model**: change an agent's `llm=` argument to any of the five named `LLM` objects defined in Step 4, or define a new one pointing at OpenAI (`gpt-4o-mini`) or another Groq model. Only the one line changes.
- **Tighten the web search budget**: reduce `max_iter` on `web_researcher_agent` or lower `max_results` in the `tavily_web_search` tool to use fewer API credits per run.
- **Adjust word-count limits**: edit the numeric values in `report_inputs` (Step 5). All eight limits — including `final_word_count` for the executive summary — flow automatically into every task template.
- **Tune writing style**: edit `skills/SKILL.md` — that file is read fresh each time you run Step 6, so any changes flow straight into the Fact-Checker & Humanizing Editor's `backstory` on the next run.
- **Add more dimensions**: extend the `multi_dimension_task` description with additional sub-topics (e.g., gender, health, or trade) — no agent changes needed.
- **Switch to hierarchical execution**: replace `Process.sequential` with `Process.hierarchical` in the `Crew(...)` call and add a `manager_llm=` argument to let a manager agent decide the running order dynamically.